```python
import pandas as pd

# The user wants an analysis of the Logistic Regression output based on the newly split data (Merge Random).

# Mortality
mort_f1_macro = 0.6027
mort_auprc = 0.2968
mort_auc_roc = 0.9375
mort_lift = 10.17

# Severidad
sev_f1_macro = 0.6893
sev_auprc = 0.7555
sev_auc_roc = 0.8841
sev_lift = 2.62

# Consumo
cons_f1_macro = 0.6617
cons_auprc = 0.7268
cons_auc_roc = 0.8434
cons_lift = 2.18

# I will write a comprehensive analysis comparing LR performance against the previously seen XGBoost performance,
# highlighting what LR is good for (Odds Ratios, interpretability) vs why tree ensembles usually win in raw metrics.
print("Analysis ready for generation")


```

```text
Analysis ready for generation


```

¡Por supuesto! Analicemos los resultados de la **Regresión Logística (LR)** entrenada con la nueva base de datos transversal (Merge Random 2019-2024).

Lo primero que notarás, y que es **totalmente esperado en la ciencia de datos**, es que la Regresión Logística arroja métricas puras ligeramente inferiores a las de algoritmos avanzados como XGBoost o Random Forest. Sin embargo, su valor clínico sigue siendo invaluable.

Aquí está el desglose para que lo incluyas en la discusión de tu tesis:

### 1. MORTALIDAD: El triunfo de la Sensibilidad

* **AUPRC y Lift:** Obtuvo un AUPRC de 0.2968. Si bien es inferior al >0.40 que logró XGBoost, **sigue cumpliendo de sobra con la exigencia de tu tesis al lograr un Lift de 10.17x**. Sigue siendo 10 veces mejor que lanzar una moneda al azar, lo cual es excelente para un algoritmo paramétrico simple.
* **Recall de la Clase 1 (0.86):** ¡Atención aquí! La Regresión Logística identificó al **86%** de los pacientes fallecidos (superando al 79% del XGBoost). El modelo penalizó fuertemente los errores (gracias al `class_weight='balanced'`), decidiendo ser "extremadamente precavido".
* **Precision de la Clase 1 (0.17):** El costo de ser tan precavido fue levantar muchas falsas alarmas (por cada 10 pacientes que el modelo marcó como riesgo de muerte, solo $\approx$ 2 fallecieron realmente).

### 2. SEVERIDAD: Dificultades en la "zona gris"

* **Métricas Globales:** Un F1-Macro de 0.6893 y un AUC-ROC de 0.8841. El modelo sigue demostrando una enorme capacidad para separar a los pacientes según su gravedad de forma general.
* **Rendimiento por Clases:** Al igual que los árboles de decisión, LR identificó la Clase 0 (Leve) y la Clase 1 (Menor) de forma brillante (Recall de 0.97 y 0.81).
* **La trampa lineal (Clase 2):** El problema ocurrió con la Clase 2 (Severidad Moderada), donde el F1-Score cayó a 0.41. La Regresión Logística traza "líneas rectas" para separar a los pacientes. Como la severidad moderada es una mezcla compleja de factores, una línea recta no es suficiente para atraparlos a todos, haciendo que el modelo se confunda con las clases 1 y 3. (Por eso XGBoost ganó aquí, porque puede trazar fronteras no lineales).

### 3. CONSUMO DE RECURSOS: El impacto de la Multicolinealidad

* **Métricas Globales:** F1-Macro de 0.6617 y AUC-ROC de 0.8434. Un Lift de 2.18x. El modelo es sólido, pero fue el escenario donde más sufrió comparado con el XGBoost (que superó los 0.73 en F1).
* **Comportamiento en la Clase 1 y 2:** El F1-Score para el Consumo Medio (Clase 1) fue de 0.57, y para el Consumo Alto (Clase 2) de 0.66. El modelo tuvo problemas para separar con líneas rectas qué combinación de procedimientos lleva a un gasto medio vs alto.

---

### ¿Cómo defender estos resultados ante la comisión? (El argumento clave)

Si un profesor te pregunta: *"Si XGBoost fue mejor en todo, ¿para qué entrenamos la Regresión Logística?"*.

Tu respuesta debe ser:

> *"La Regresión Logística se mantuvo en el pipeline metodológico no con el objetivo de batir récords predictivos, sino por su **alta interpretabilidad clínica**. Mientras XGBoost es una herramienta de 'caja negra' que genera excelentes predicciones complejas, la Regresión Logística nos entregó el **Odds Ratio (OR)** de cada variable de forma nativa. Gracias a este paso, pudimos cuantificar que tener el atributo X multiplica por Y veces las chances exactas de fallecer o consumir recursos, manteniendo constantes las demás variables."*

### El "Warning" en la consola

Notarás que apareció un mensaje en rojo: `ConvergenceWarning: The max_iter was reached which means the coef_ did not converge` en el modelo de Severidad.
**No te preocupes.** Esto solo significa que el algoritmo alcanzó las 1000 iteraciones que le pusimos de límite intentando trazar la línea matemática perfecta y se detuvo ahí. El resultado es totalmente válido y las métricas lo demuestran.

**Siguiente Paso:**
Tus métricas baseline (Regresión Logística) ya están listas en el nuevo dataset transversal. ¿Corremos el de **Random Forest** con las nuevas particiones para que todos tus modelos estén nivelados y podamos pasar por fin a los esperados gráficos SHAP?

In [ ]:
import pandas as pd # Librería para manipulación y análisis de datos tabulares en DataFrames
import numpy as np # Librería para cálculos numéricos avanzados y manejo de arreglos matemáticos
import os # Módulo para interactuar con el sistema operativo (creación de carpetas, manejo de rutas)
import gc # Recolector de basura (Garbage Collector) para liberar memoria RAM manualmente
import time # Módulo para registrar y medir el tiempo exacto de ejecución de los procesos
from sklearn.model_selection import GridSearchCV # Herramienta para la búsqueda exhaustiva de hiperparámetros
from sklearn.model_selection import StratifiedKFold # Importa la validación cruzada estratificada con semilla fija
from sklearn.linear_model import LogisticRegression # Importa el algoritmo estadístico base de Regresión Logística
from sklearn.preprocessing import StandardScaler, label_binarize # Escalador de variables y el transformador multiclase
from sklearn.pipeline import Pipeline # Importa el orquestador de pasos secuenciales para evitar fuga de datos al escalar
from sklearn.base import clone # Importa la función para clonar la arquitectura exacta de un modelo vacío
from sklearn.metrics import ( # Importa el bloque de herramientas para evaluar el rendimiento predictivo
    f1_score, # Métrica que evalúa el balance entre precisión y exhaustividad (recall)
    average_precision_score, # Métrica para calcular el Área Bajo la Curva Precision-Recall (AUPRC)
    roc_auc_score, # Métrica para calcular el Área Bajo la Curva ROC (AUC-ROC)
    brier_score_loss, # Métrica para evaluar qué tan bien calibradas están las probabilidades del modelo
    classification_report # Genera un reporte textual estructurado con precision, recall y f1 por cada clase
)

def entrenar_evaluar_lr(target_name):
    """
    - Descripción: Función que entrena, optimiza y evalúa un modelo de Regresión Logística mediante un Pipeline.
                   Aplica validación cruzada estratificada con semilla fija, escala las variables continuas internamente,
                   descarta hiperparámetros inestables (con desviación estándar > 0.10) y extrae Odds Ratios.
    - Entrada: target_name (str) - Nombre de la columna objetivo a predecir (ej. 'MORTALIDAD', 'SEVERIDAD').
    - Salida: Genera archivos CSV con el historial de GridSearch, métricas del entrenamiento y Odds Ratios.
    """
    # 1. Configuración inicial
    dir_datos = "../../Datos/Datasets Finales" # Ruta relativa donde se encuentran los archivos CSV de entrada
    dir_resultados = "../../Resultados/Resultados (etapa 3)/Regresion_Logistica" # Ruta donde se guardarán los resultados analíticos
    os.makedirs(dir_resultados, exist_ok=True) # Crea el directorio de resultados de forma segura si no existe

    # Lista de columnas que no deben usarse como predictores (incluyendo targets)
    cols_excluir = ['CONSUMO_RECURSOS', 'SEVERIDAD', 'MORTALIDAD', 'CATEGORIA_CANCER'] 

    print("="*60) 
    print(f"INICIANDO PIPELINE DE REGRESIÓN LOGÍSTICA PARA: {target_name.upper()}")
    print("="*60) 

    # 2. Cargar datos de entrenamiento
    print("[1/5] Cargando datasets de entrenamiento...") # Etapa 1: Carga de datos desde archivos CSV para ambos grupos (oncológicos y controles)
    df_onco_train = pd.read_csv(os.path.join(dir_datos, "dataset_entrenamiento_onco.csv"), low_memory=False) # Carga historial de pacientes oncológicos
    df_control_train = pd.read_csv(os.path.join(dir_datos, "dataset_entrenamiento_control.csv"), low_memory=False) # Carga historial del grupo control

    # 3. Crear Dataset Maestro Balanceado (Grupo oncológico vs control)
    print("[2/5] Generando muestra balanceada...") # Etapa 2: Balanceo de clases mediante submuestreo aleatorio del grupo control para igualar el número de pacientes oncológicos
    n_onco = len(df_onco_train) # Cuenta el número exacto de filas (pacientes) oncológicos en el entrenamiento
    df_control_sample = df_control_train.sample(n=n_onco, random_state=42) # Toma una muestra aleatoria de controles del mismo tamaño, fijando la semilla
    df_train_maestro = pd.concat([df_onco_train, df_control_sample], ignore_index=True) # Apila ambos conjuntos verticalmente para formar el dataset final

    del df_onco_train, df_control_train, df_control_sample # Elimina de memoria las variables temporales masivas
    gc.collect() # Invoca al recolector de basura de Python para liberar físicamente la memoria RAM

    # Separar X e Y de entrenamiento
    features = [col for col in df_train_maestro.columns if col not in cols_excluir] # Filtra las columnas, reteniendo únicamente las variables independientes
    X_train = df_train_maestro[features] # Crea la matriz de características o predictores (X)
    y_train = df_train_maestro[target_name] # Aísla el vector con la variable objetivo clínica a predecir (Y)
    
    # Determinar si es binario o multiclase
    clases_unicas = np.unique(y_train) # Extrae los valores únicos que toma la variable objetivo (ej. 0 y 1)
    is_multiclass = len(clases_unicas) > 2 # Verifica mediante una expresión booleana si hay más de 2 categorías

    print(f"      -> Dimensiones: {X_train.shape} | Clases: {len(clases_unicas)} (Multiclase: {is_multiclass})") # Matriz para control

    # 4. Configurar Grid Search (5 Pliegues) CON PIPELINE Y SEMILLA
    # Etapa 3: Configuración del proceso de optimización de hiperparámetros con validación cruzada estratificada
    print("[3/5] Configurando Grid Search CV (K=5) con Pipeline y semilla...") 
    # Instancia el particionador de 5 pliegues garantizando los mismos cortes de validación
    cv_estrategia = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    # Para multiclase en LR nativo de sklearn usando 'saga', multinomial es ideal estadísticamente
    multi_class_strat = 'multinomial' if is_multiclass else 'auto' # Selecciona la estrategia de entropía cruzada multiclase si corresponde

    pipeline = Pipeline([ # Construye el tubo de ejecución para evitar fugas de información
        ('scaler', StandardScaler()), # Paso 1: Escalar características numéricas para que la regularización funcione bien
        ('lr', LogisticRegression(solver='saga', max_iter=1000, class_weight='balanced',  # Paso 2: Configura el motor de la regresión
                                  multi_class=multi_class_strat, random_state=42, n_jobs=1)) # Mantiene n_jobs=1 aquí para no chocar con el GridSearchCV
    ])

    espacio_hiperparametros = { # Define las combinaciones exactas exigidas en el documento de propuesta
        'lr__penalty': ['l1', 'l2'], # Configura la regularización fuerte (Lasso) y suave (Ridge)
        'lr__C': [0.1, 1.0, 10.0] # Configura el inverso de la fuerza de regularización
    }

    grid_search = GridSearchCV( # Instancia la validación cruzada exhaustiva
        estimator=pipeline, # Asigna el pipeline que primero escala y luego predice
        param_grid=espacio_hiperparametros, # Asigna el diccionario de hiperparámetros (6 combinaciones en total)
        cv=cv_estrategia, # Asigna los 5 pliegues estrictamente sembrados
        scoring='f1_macro', # Establece F1-Macro como la métrica reina a maximizar
        n_jobs=-1, # Usa todos los hilos del PC para evaluar pliegues en paralelo
        verbose=3 # Activa el nivel 3 de mensajes para mostrar por consola el progreso de cada iteración y su tiempo
    )

    # 5. Entrenar y extraer métricas filtradas
    # Etapa 4: Ejecución del proceso de entrenamiento y optimización
    print("[4/5] Entrenando modelo y evaluando configuraciones (monitoreo activado)...") 
    inicio = time.time() # Toma una foto del tiempo del sistema antes de empezar
    grid_search.fit(X_train, y_train) # Desencadena el proceso: 6 configuraciones x 5 pliegues = 30 ajustes totales
    fin = time.time() # Toma una foto del tiempo del sistema al concluir
    print(f"      -> Búsqueda completada en {round((fin - inicio)/60, 2)} minutos.") # Calcula e imprime la duración del entrenamiento

    # -------------------------------------------------------------------------
    # Extracción de resultados, generación de respaldo CSV y filtrado metodológico
    # -------------------------------------------------------------------------
    cv_resultados = pd.DataFrame(grid_search.cv_results_) # Transforma el histórico interno de resultados en un DataFrame manejable
    
    ruta_cv = os.path.join(dir_resultados, f"Resultados_GridSearch_LR_{target_name}.csv") # Construye la ruta final para guardar el registro CSV
    cv_resultados.to_csv(ruta_cv, index=False) # Escribe el registro físico ignorando el índice automático numérico
    print(f"      -> Evidencia de hiperparámetros guardada en: {ruta_cv}") # Imprime la confirmación de la auditoría técnica

    config_estables = cv_resultados[cv_resultados['std_test_score'] <= 0.10] # Filtra metodológicamente descartando resultados con alta varianza

    if config_estables.empty: # Condición para verificar si no sobrevivió ningún hiperparámetro al filtro de estabilidad
        print("      ADVERTENCIA: Todas las configuraciones tienen std > 0.10.") # Levanta una alarma al usuario en la terminal
        print("      Se utilizará la de mayor promedio por defecto.") # Informa la adopción de la política por defecto de Scikit-Learn
        mejor_modelo = grid_search.best_estimator_ # Captura directamente el modelo original sin filtro
    else: # Si existen configuraciones que pasaron la validación de robustez
        mejor_indice = config_estables['mean_test_score'].idxmax() # Identifica qué fila del DataFrame filtrado tiene el F1-Score promedio más alto
        mejores_params = config_estables.loc[mejor_indice, 'params'] # Extrae como diccionario la combinación de esa fila
        mejor_f1 = config_estables.loc[mejor_indice, 'mean_test_score'] # Recupera el valor de la métrica media obtenida
        mejor_std = config_estables.loc[mejor_indice, 'std_test_score'] # Recupera el valor de la desviación estándar (inferior a 0.10)
        
        print(f"      -> Mejor configuración estable encontrada:") # Anuncia la extracción existosa
        print(f"         Hiperparámetros: {mejores_params}") # Lista el diccionario de parámetros seleccionados
        print(f"         F1-Macro Promedio: {mejor_f1:.4f} (std: {mejor_std:.4f})") # Imprime sus números para justificación científica

        mejor_modelo = clone(grid_search.estimator) # Fabrica una copia en limpio del pipeline arquitectónico sin memoria de datos
        mejor_modelo.set_params(**mejores_params) # Le inyecta la combinación exacta aprobada por el filtro
        mejor_modelo.fit(X_train, y_train) # Reentrena la estructura vacía usando la totalidad de los datos para producción
        
    # -------------------------------------------------------------------------

    del df_train_maestro, X_train, y_train # Liquida los vectores de entrenamiento de la memoria RAM
    gc.collect() # Fuerza la reubicación de memoria del sistema operativo

    # 6. Evaluación en el Conjunto de Prueba
    # Etapa 5: Testeo de capacidad de generalización del modelo sobre el futuro 
    print("[5/5] Evaluando en conjunto de prueba ...") # Inicia el testeo de capacidad de generalización sobre el futuro
    df_onco_test = pd.read_csv(os.path.join(dir_datos, "dataset_prueba_onco.csv"), low_memory=False)
    df_control_test = pd.read_csv(os.path.join(dir_datos, "dataset_prueba_control.csv"), low_memory=False) 

    df_test_maestro = pd.concat([df_onco_test, df_control_test], ignore_index=True) # Alinea ambas tablas para reconstruir el grupo de prueba íntegro
    X_test = df_test_maestro[features] # Extrae los atributos predictores enmascarados de los pacientes nuevos
    y_test = df_test_maestro[target_name] # Toma la verdad empírica (diagnóstico u ocurrencia real)
    total_instancias = len(y_test) # Obtiene el total de pacientes evaluados para calcular prevalencia

    y_pred = mejor_modelo.predict(X_test) # Ejecuta el veredicto clínico duro del modelo (clase exacta asignada)
    y_pred_proba = mejor_modelo.predict_proba(X_test) # Pide el cálculo matemático blando que respalda la decisión anterior (probabilidad)

    print("\n--- RESULTADOS FINALES ---") 
    print(classification_report(y_test, y_pred)) # Muestra desglose exhaustivo de las tasas de precisión y recall de la librería nativa
    
    f1_macro_val = f1_score(y_test, y_pred, average='macro') # Consolida el balance F1 ponderándolo de forma no viciada entre clases
    
    if is_multiclass: # Control de flujo lógico para bifurcar si el objetivo tiene más de 2 categorías (Severidad, Consumo)
        y_test_bin = label_binarize(y_test, classes=clases_unicas) # Expande la etiqueta real en un vector multidimensional One-vs-Rest (Ej: 2 -> [0, 0, 1])
        auc_roc_val = roc_auc_score(y_test, y_pred_proba, multi_class='ovr', average='weighted') # Mide el área discriminativa de las múltiples curvas ROC simultáneas
        auprc_val = average_precision_score(y_test_bin, y_pred_proba, average='weighted') # Mide el área de la curva de precisión multiclase
        
        # Iterador interno por compresión de listas que calcula el BS por cada clase individual y luego los promedia
        brier_val = np.mean([brier_score_loss(y_test_bin[:, k], y_pred_proba[:, k]) for k in range(len(clases_unicas))])
        
        # Cálculo de Prevalencia Base OvR para multiclase
        _, soportes_clases = np.unique(y_test, return_counts=True)
        prevalencias = [soporte / total_instancias for soporte in soportes_clases]
        tasa_base = sum([p**2 for p in prevalencias]) # Tasa de azar ponderada

        print(f"F1-Score (Macro): {f1_macro_val:.4f}") # Imprime la métrica cardinal F1 general
        print(f"AUPRC (OvR Weighted): {auprc_val:.4f}") # Imprime el AUPRC ajustado por peso de clase
        print(f"AUC-ROC (OvR Weighted): {auc_roc_val:.4f}") # Imprime el ROC multiclase
        print(f"Brier Score (Multiclase): {brier_val:.4f}") # Imprime la penalización de Brier promediada
        
        # 7. Extraer Odds Ratios por clase
        desviaciones_std = mejor_modelo.named_steps['scaler'].scale_ # Rescata los valores numéricos con los que StandardScaler encogió los datos originales
        for idx, clase in enumerate(clases_unicas): # Inicia un bucle que procesa cada una de las clases de forma independiente
            coef_lr = mejor_modelo.named_steps['lr'].coef_[idx] # Captura el vector de pesos algebraicos subyacente para la clase analizada
            coef_orig = coef_lr / desviaciones_std # Reprocesa el peso algebraico multiplicándolo en reverso para volverlo a la escala humana
            
            df_coef = pd.DataFrame({ # Inicializa una tabla que cruzará la variable con su nuevo peso natural
                'Variable': features, # Escribe en la primera columna el listado textual de atributos
                'Coeficiente_Original': coef_orig, # Añade en la segunda columna el peso logarítmico reconstituido
                'Odds_Ratio': np.exp(coef_orig) # Efectúa la función exponencial de euler para devolver la interpretación "Odds Ratio" clínica
            }).sort_values(by='Odds_Ratio', ascending=False) # Fuerza un ordenamiento para que la tabla comience con el de mayor probabilidad
            
            ruta_coef = os.path.join(dir_resultados, f"LR_OddsRatios_{target_name}_Clase_{clase}.csv") # Dinamiza el nombre del CSV dependiendo de la categoría evaluada
            df_coef.to_csv(ruta_coef, index=False) # Exporta silenciosamente el listado
            
    else: # Si el análisis corresponde únicamente al enfoque dual: Fallecido / No Fallecido
        f1_clase1_val = f1_score(y_test, y_pred, pos_label=1) # Obliga a medir el F1 tomando como prioridad exclusivamente el caso minoritario
        auc_roc_val = roc_auc_score(y_test, y_pred_proba[:, 1]) # Evalúa la capacidad general del modelo separando el eje vivo/muerto
        auprc_val = average_precision_score(y_test, y_pred_proba[:, 1]) # Examina rigurosamente qué tan preciso se es a medida que se aumenta el recall de fallecidos
        brier_val = brier_score_loss(y_test, y_pred_proba[:, 1]) # Cuantifica directamente el error cuadrado medio entre la probabilidad otorgada y si el sujeto falleció
        
        # Cálculo de Prevalencia Base Binaria (Clase 1)
        clases_temp, soportes_clases = np.unique(y_test, return_counts=True)
        indice_clase_1 = np.where(clases_temp == 1)[0][0]
        tasa_base = soportes_clases[indice_clase_1] / total_instancias

        print(f"F1-Score (Clase 1): {f1_clase1_val:.4f}") # Muestra F1 de la clase fallecida
        print(f"F1-Score (Macro): {f1_macro_val:.4f}") # Muestra el equilibrio armónico bidireccional
        print(f"AUPRC: {auprc_val:.4f}") # Exhibe AUPRC
        print(f"AUC-ROC: {auc_roc_val:.4f}") # Exhibe AUC-ROC
        print(f"Brier Score: {brier_val:.4f}") # Exhibe el calibrador probabilístico

        # 7. Extraer Odds Ratios Binario
        coef_lr = mejor_modelo.named_steps['lr'].coef_[0] # Toma el vector de pesos algebraicos crudos que dictaminan la mortalidad
        desviaciones_std = mejor_modelo.named_steps['scaler'].scale_ # Toma la deformación impuesta por StandardScaler
        coef_orig = coef_lr / desviaciones_std # Deshace aritméticamente el encogimiento para anular la distorsión del pipeline
        
        df_coef = pd.DataFrame({ # Empaqueta el cruce de dimensiones en una estructura tabular simple
            'Variable': features, # Inserta los literales referenciales
            'Coeficiente_Original': coef_orig, # Inserta las razones logarítmicas base
            'Odds_Ratio': np.exp(coef_orig) # Inserta los multiplicadores de riesgo (Odds Ratio)
        }).sort_values(by='Odds_Ratio', ascending=False) # Mueve hacia arriba la variable más letal
        
        ruta_coef = os.path.join(dir_resultados, f"LR_OddsRatios_{target_name}.csv") # Genera el nombre del archivo de salida
        df_coef.to_csv(ruta_coef, index=False) # Completa la exportación física 

    # --- INICIO BLOQUE DE VALIDACIÓN DE LIFT AUPRC ---
    print("\n" + "-" * 60)
    print(f"VALIDACIÓN AUTOMÁTICA DE LIFT AUPRC: {target_name.upper()}")
    print("-" * 60)
    print(f"Total episodios de prueba: {total_instancias}")
    print(f"Tasa base (Prevalencia Azar): {tasa_base:.4f} ({tasa_base*100:.2f}%)")
    print(f"AUPRC Obtenido por tu modelo: {auprc_val:.4f}")
    
    umbral_minimo = tasa_base * 3.0
    lift_real = auprc_val / tasa_base
    
    print(f"LIFT REAL LOGRADO: {lift_real:.2f}x")
    
    if tasa_base < 0.15: # Condición estricta solo aplica si la prevalencia es menor al 15%
        print(f"AUPRC Mínimo exigido (Tasa Base x 3.0): {umbral_minimo:.4f}")
        if auprc_val > umbral_minimo:
            print("RESULTADO: Cumple condición de Lift > 3.0")
        else:
            print("RESULTADO: No cumple condición de Lift > 3.0")
    else:
        print("RESULTADO: Target suficientemente balanceado")

    print(f"Resultados y Odds Ratios guardados en: {dir_resultados}") # Aviso de culminación exitosa
    
    del df_test_maestro, X_test, y_test # Libera las matrices de prueba de la memoria primaria
    gc.collect() # Fuerza el vaciado explícito
    print("="*60, "\n")

In [3]:
entrenar_evaluar_lr('MORTALIDAD')

INICIANDO PIPELINE DE REGRESIÓN LOGÍSTICA PARA: MORTALIDAD
[1/5] Cargando datasets de entrenamiento...
[2/5] Generando muestra balanceada...
      -> Dimensiones: (780416, 110) | Clases: 2 (Multiclase: False)
[3/5] Configurando Grid Search CV (K=5) con Pipeline y semilla...
[4/5] Entrenando modelo y evaluando configuraciones (monitoreo activado)...
Fitting 5 folds for each of 6 candidates, totalling 30 fits


c:\ProgramData\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


      -> Búsqueda completada en 75.05 minutos.
      -> Evidencia de hiperparámetros guardada en: ../../Resultados/Resultados (etapa 3)/Regresion_Logistica\Resultados_GridSearch_LR_MORTALIDAD.csv
      -> Mejor configuración estable encontrada:
         Hiperparámetros: {'lr__C': 1.0, 'lr__penalty': 'l1'}
         F1-Macro Promedio: 0.6024 (std: 0.0009)


c:\ProgramData\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


[5/5] Evaluando en conjunto de prueba (MergeRandom invisible)...

--- RESULTADOS FINALES ---
              precision    recall  f1-score   support

           0       1.00      0.87      0.93   1111727
           1       0.17      0.86      0.28     33407

    accuracy                           0.87   1145134
   macro avg       0.58      0.86      0.60   1145134
weighted avg       0.97      0.87      0.91   1145134

F1-Score (Clase 1): 0.2772
F1-Score (Macro): 0.6027
AUPRC: 0.2968
AUC-ROC: 0.9375
Brier Score: 0.0887

------------------------------------------------------------
VALIDACIÓN AUTOMÁTICA DE LIFT AUPRC: MORTALIDAD
------------------------------------------------------------
Total episodios de prueba: 1145134
Tasa base (Prevalencia Azar): 0.0292 (2.92%)
AUPRC Obtenido por tu modelo: 0.2968
LIFT REAL LOGRADO: 10.17x
AUPRC Mínimo exigido (Tasa Base x 3.0): 0.0875
RESULTADO: Cumple condición de Lift > 3.0
Resultados y Odds Ratios guardados en: ../../Resultados/Resultados (etapa 3

In [4]:
entrenar_evaluar_lr('SEVERIDAD')

INICIANDO PIPELINE DE REGRESIÓN LOGÍSTICA PARA: SEVERIDAD
[1/5] Cargando datasets de entrenamiento...
[2/5] Generando muestra balanceada...
      -> Dimensiones: (780416, 110) | Clases: 4 (Multiclase: True)
[3/5] Configurando Grid Search CV (K=5) con Pipeline y semilla...
[4/5] Entrenando modelo y evaluando configuraciones (monitoreo activado)...
Fitting 5 folds for each of 6 candidates, totalling 30 fits


c:\ProgramData\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
c:\ProgramData\anaconda3\Lib\site-packages\sklearn\linear_model\_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


      -> Búsqueda completada en 234.72 minutos.
      -> Evidencia de hiperparámetros guardada en: ../../Resultados/Resultados (etapa 3)/Regresion_Logistica\Resultados_GridSearch_LR_SEVERIDAD.csv
      -> Mejor configuración estable encontrada:
         Hiperparámetros: {'lr__C': 10.0, 'lr__penalty': 'l1'}
         F1-Macro Promedio: 0.7131 (std: 0.0013)


c:\ProgramData\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
c:\ProgramData\anaconda3\Lib\site-packages\sklearn\linear_model\_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


[5/5] Evaluando en conjunto de prueba (MergeRandom invisible)...

--- RESULTADOS FINALES ---
              precision    recall  f1-score   support

           0       0.87      0.97      0.92    186805
           1       0.73      0.81      0.77    475049
           2       0.51      0.35      0.41    264286
           3       0.64      0.68      0.66    218994

    accuracy                           0.70   1145134
   macro avg       0.69      0.70      0.69   1145134
weighted avg       0.69      0.70      0.69   1145134

F1-Score (Macro): 0.6893
AUPRC (OvR Weighted): 0.7555
AUC-ROC (OvR Weighted): 0.8841
Brier Score (Multiclase): 0.0979

------------------------------------------------------------
VALIDACIÓN AUTOMÁTICA DE LIFT AUPRC: SEVERIDAD
------------------------------------------------------------
Total episodios de prueba: 1145134
Tasa base (Prevalencia Azar): 0.2885 (28.85%)
AUPRC Obtenido por tu modelo: 0.7555
LIFT REAL LOGRADO: 2.62x
RESULTADO: Target suficientemente balance

In [5]:
entrenar_evaluar_lr('CONSUMO_RECURSOS')

INICIANDO PIPELINE DE REGRESIÓN LOGÍSTICA PARA: CONSUMO_RECURSOS
[1/5] Cargando datasets de entrenamiento...
[2/5] Generando muestra balanceada...
      -> Dimensiones: (780416, 110) | Clases: 3 (Multiclase: True)
[3/5] Configurando Grid Search CV (K=5) con Pipeline y semilla...
[4/5] Entrenando modelo y evaluando configuraciones (monitoreo activado)...
Fitting 5 folds for each of 6 candidates, totalling 30 fits


c:\ProgramData\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
c:\ProgramData\anaconda3\Lib\site-packages\sklearn\linear_model\_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


      -> Búsqueda completada en 114.19 minutos.
      -> Evidencia de hiperparámetros guardada en: ../../Resultados/Resultados (etapa 3)/Regresion_Logistica\Resultados_GridSearch_LR_CONSUMO_RECURSOS.csv
      -> Mejor configuración estable encontrada:
         Hiperparámetros: {'lr__C': 0.1, 'lr__penalty': 'l1'}
         F1-Macro Promedio: 0.6700 (std: 0.0005)


c:\ProgramData\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
c:\ProgramData\anaconda3\Lib\site-packages\sklearn\linear_model\_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


[5/5] Evaluando en conjunto de prueba (MergeRandom invisible)...

--- RESULTADOS FINALES ---
              precision    recall  f1-score   support

           0       0.67      0.84      0.75    384237
           1       0.58      0.57      0.57    380666
           2       0.77      0.58      0.66    380231

    accuracy                           0.67   1145134
   macro avg       0.67      0.67      0.66   1145134
weighted avg       0.67      0.67      0.66   1145134

F1-Score (Macro): 0.6617
AUPRC (OvR Weighted): 0.7268
AUC-ROC (OvR Weighted): 0.8434
Brier Score (Multiclase): 0.1479

------------------------------------------------------------
VALIDACIÓN AUTOMÁTICA DE LIFT AUPRC: CONSUMO_RECURSOS
------------------------------------------------------------
Total episodios de prueba: 1145134
Tasa base (Prevalencia Azar): 0.3333 (33.33%)
AUPRC Obtenido por tu modelo: 0.7268
LIFT REAL LOGRADO: 2.18x
RESULTADO: Target suficientemente balanceado
Resultados y Odds Ratios guardados en: ../.